In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [1]:
import yaml

from sim.drive_simulator import CarSim

from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)


def start(mission: MissionBase):
    sim = CarSim(prop, mission, debug_log=True)
    complete = sim.run()
    if complete:
        SimDrawer(sim).show()
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        return goaled
    else:
        return False

c:\Users\k-ueda\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class Mission4(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=80)
        self.goals = [
            GoalCircle((2, 2), 0.2, should_stop=True),
            GoalCircle((2.5, 0.0), 0.2, should_stop=False),
        ]
        self.initial_xy = (2.5, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=2, y=2, name="original"),
                ### 標識を追加するにはここから下を書き換える
                Sign(x=1.4, y=0.9, name="stop"),
                Sign(x=0.8, y=0.8, name="warn"),
                ### 標識を追加するにはここより上を書き換える
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, wait, search, auto, **kwargs):
        rotate(45)
        # ######## ここから下にプログラムを書こう
        # mode = 0
        # while alive():
        #     if mode == 0:
        #         # コース内で自動走行し、標識(original)を見つけた場合に標識の真上に行き１秒停車するモード
        #         pos = search(name="original")
        #         if pos is None:
        #             auto(v=0.2)
        #         else:
        #             if pos.theta > 5:
        #                 rotate(w=45)
        #             elif pos.theta < -5:
        #                 rotate(w=-45)
        #             else:
        #                 move(v=0.2)
        #                 move(v=0.2, t=pos.x / 0.2)
        #                 wait()
        #                 move(v=0, t=1)
        #                 wait()
        #                 mode = 1
        #     elif mode == 1:
        #         # 左回転し、標識(stop)を見つけた場合に標識の真上に行くモード
        #         pos = search(name="stop")
        #         if pos is None:
        #             rotate(w=45)
        #         else:
        #             if pos.theta > 5:
        #                 rotate(w=45)
        #             elif pos.theta < -5:
        #                 rotate(w=-45)
        #             else:
        #                 move(v=0.2)
        #                 move(v=0.2, t=pos.x / 0.2)
        #                 wait()
        #                 mode = 2
        #     elif mode == 2:
        #         # 右回転し、標識(warn)を見つけた場合に標識の方を向いた後で自動走行を開始するモード
        #         pos = search(name="warn")
        #         if pos is None:
        #             rotate(w=-45)
        #         else:
        #             rotate(w=pos.theta, t=1)
        #             wait()
        #             auto(v=0.2)
        #             mode = 0

        # ######## ここより上にプログラムを書こう


"成功" if start(Mission4()) else "失敗"

drive_dt=0.031, detect_dt=0.061, throttle=20
[0.000] put speed command v=0.2, w=auto, t=None, auto=lane_center
[0.000] recv command: SpeedCommand delay=0.000
[0.000] changed v=0.200, auto=lane_center:dt=0.5


c:\data\git\drive_sim\sim\drive_simulator.py:567: RuntimeWarning: invalid value encountered in divide
  (np.sin(yaw_end_patterns) - np.sin(yaw_start_patterns)) / k_patterns,
c:\data\git\drive_sim\sim\drive_simulator.py:573: RuntimeWarning: invalid value encountered in divide
  (np.cos(yaw_start_patterns) - np.cos(yaw_end_patterns)) / k_patterns,


end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
end calc auto_w
[8.573] command_func finished
[8.573] simulation_func finished
    takes 22.860s
    ideal 0.429s
Trajectory points : 280
Active threads : 6


  9%|▊         | 8/94 [00:00<00:01, 77.74it/s]c:\data\git\drive_sim\sim\drive_simulator.py:567: RuntimeWarning: invalid value encountered in divide
  (np.sin(yaw_end_patterns) - np.sin(yaw_start_patterns)) / k_patterns,
c:\data\git\drive_sim\sim\drive_simulator.py:573: RuntimeWarning: invalid value encountered in divide
  (np.cos(yaw_start_patterns) - np.cos(yaw_end_patterns)) / k_patterns,
100%|██████████| 94/94 [00:00<00:00, 184.01it/s]


'失敗'

In [ ]:
# 複数回実行して必ず成功するかのチェック
CHECK_CNT = 10
ok = 0
for i in range(CHECK_CNT):
    mission = Mission4()
    sim = CarSim(prop, mission)
    complete = sim.run()
    if complete:
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        if goaled:
            ok += 1
        else:
            SimDrawer(sim).show()
            break
        print("-------------------------------OK:", ok)